# Using NVIDIA Nemotron-3-Super-120B with Stained Glass Output Protection on SageMaker

This sample notebook shows you how to deploy **NVIDIA-Nemotron-3-Super-120B** with Stained Glass on Amazon SageMaker, and invoke it end to end.

Stained Glass protects **both** ends of the request:
- **Input protection (`prompt_embeds`)** — the prompt is sent as embeddings, never plaintext.
- **Output protection** — the generated tokens are returned **encrypted** (tokenwise x25519 + AES-GCM). The client supplies an ephemeral public key, the server returns its public key, both derive a shared AES key, and the client decrypts the response.

Because SageMaker's `InvokeEndpoint` strips arbitrary HTTP headers, the public keys travel over the **`CustomAttributes`** field.

## Pre-requisites
1. Get this notebook from the model catalog on the SageMaker AWS Management Console.
2. Ensure the IAM role used has **AmazonSageMakerFullAccess**.

## Contents
1. AWS Setup
2. Create an endpoint
3. Output Protection helper (key exchange + decrypt)
4. Real-time inference: chat, completions, and prompt_embeds
5. Clean-up

Run one cell at a time with Shift+Enter.

## AWS Setup

In [ ]:
# cryptography provides the x25519 + AES-GCM primitives used to decrypt the Output-Protected
# response. torch is only needed to build the prompt_embeds payload.
!pip install boto3 sagemaker cryptography torch

### Helper Functions

In [ ]:
import json

import boto3
import sagemaker as sage


def assume_role(
    role_arn: str,
    boto_session: boto3.Session | None = None,
    session_name: str = "AssumeRoleSession",
    region_name: str | None = None,
) -> boto3.Session:
    """Create a session with an assumed role (useful outside SageMaker notebooks)."""
    boto_session = boto_session if boto_session is not None else boto3.Session()
    region_name = boto_session.region_name if region_name is None else region_name
    sts_client = boto_session.client("sts")
    response = sts_client.assume_role(RoleArn=role_arn, RoleSessionName=session_name)
    credentials = response["Credentials"]
    return boto3.Session(
        aws_access_key_id=credentials["AccessKeyId"],
        aws_secret_access_key=credentials["SecretAccessKey"],
        aws_session_token=credentials["SessionToken"],
        region_name=region_name,
    )

### AWS Constants

In [ ]:
############# You MUST modify these constants

# The profile name for your AWS account
AWS_PROFILE = "default"

# The execution role which has `AmazonSageMakerFullAccess` permissions
EXECUTION_ROLE = "arn:aws:iam::YOUR-AWS-ACCOUNT:role/service-role/ROLE-NAME-WITH-AmazonSageMakerFullAccess-PERMISSIONS"

############# You MAY modify these constants if you wish

# The name of the model and endpoint
MODEL_NAME = "sgt-nemotron-3-super-120b"

In [ ]:
############# You MUST set this to the Nemotron model-package ARN from your Marketplace listing.

MARKETPLACE_MODEL_PACKAGE_ARN = (
    "arn:aws:sagemaker:us-east-1:600627338455:model-package/NVIDIA-Nemotron-3-Super-120B-A12B-FP8-with-vllm-prompt-embeds"
)
# Mapping for Model Packages (initially only us-east-1 is supported)
MODEL_PACKAGE_MAP = {
    "us-east-1": MARKETPLACE_MODEL_PACKAGE_ARN,
}

# region_name is explicit since not every AWS profile has a default region configured.
SESSION = boto3.Session(profile_name=AWS_PROFILE, region_name="us-east-1")
execution_session = SESSION
# Useful if using outside of SageMaker notebooks:
# execution_session = assume_role(EXECUTION_ROLE, boto_session=SESSION)

region = SESSION.region_name
if region not in MODEL_PACKAGE_MAP:
    raise ValueError("UNSUPPORTED REGION")

model_package_arn = MODEL_PACKAGE_MAP[region]

### Create the SageMaker Runtime Client

In [ ]:
sagemaker_session = sage.Session(boto_session=execution_session)
role = sage.get_execution_role(sagemaker_session=sagemaker_session)

runtime_sm_client = sagemaker_session.boto_session.client("runtime.sagemaker")

## 2. Create an endpoint

Nemotron-3-Super-120B (FP8, ~120 GB) needs a multi-GPU node. The serving image auto-sizes `--tensor-parallel-size` to the instance's GPU count. For the list of supported `instance_type` values, check the model package's supported instance types (visible on the AWS Marketplace listing page or the SageMaker console) — `ml.g7e.24xlarge` below is just a reasonable default. See the SageMaker [real-time inference docs](https://docs.aws.amazon.com/sagemaker/latest/dg/deploy-model.html) for more.

In [ ]:
ENDPOINT_DEPLOYMENT_CONFIGURATION = {
    "initial_instance_count": 1,
    "endpoint_name": MODEL_NAME,
    "instance_type": "ml.g7e.24xlarge",
    "model_data_download_timeout": 3600,
    "container_startup_health_check_timeout_in_seconds": 900,
    "inference_ami_version": "al2023-ami-sagemaker-inference-gpu-4-1",
}

# Create a deployable model from the model package.
model = sage.ModelPackage(
    role=role,
    model_package_arn=model_package_arn,
    sagemaker_session=sagemaker_session,
)

In [ ]:
# Deploy the model (multi-GPU cold start can take several minutes).
predictor = model.deploy(**ENDPOINT_DEPLOYMENT_CONFIGURATION)

## 3. Output Protection: key exchange + decrypt

Every request carries the client's ephemeral x25519 public key in `CustomAttributes`; the response returns the server's public key the same way. Both sides derive the same shared AES key, and the generated text is decrypted client-side. The helper below does this for any request body and decrypts the chosen text field.

In [ ]:
import base64
from collections.abc import Callable
from typing import Any

from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import x25519
from cryptography.hazmat.primitives.ciphers.aead import AESGCM
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

# Ciphertexts for a single choice may be concatenated with this separator.
CIPHERTEXT_SEPARATOR = "|"


def derive_shared_aes_key(private_key: x25519.X25519PrivateKey, peer_public_key: x25519.X25519PublicKey) -> bytes:
    """Derive the shared AES key the same way the Output Protection server does: x25519 + HKDF-SHA256."""
    shared_secret = private_key.exchange(peer_public_key)
    return HKDF(algorithm=hashes.SHA256(), length=32, salt=None, info=b"stainedglass").derive(shared_secret)


def decrypt_str(ciphertext: str, shared_aes_key: bytes) -> str:
    """Decrypt a base64-encoded AES-GCM ciphertext (the first 12 bytes are the nonce)."""
    aes_gcm = AESGCM(shared_aes_key)
    plaintexts = []
    for part in ciphertext.split(CIPHERTEXT_SEPARATOR):
        raw = base64.b64decode(part)
        nonce, data = raw[:12], raw[12:]
        plaintexts.append(aes_gcm.decrypt(nonce=nonce, data=data, associated_data=None).decode("utf-8"))
    return "".join(plaintexts)


def invoke_and_decrypt(body: dict[str, Any], ciphertext_of: Callable[[dict], str]) -> dict:
    """Invoke the Output-Protected endpoint and print the decrypted text of each choice.

    `ciphertext_of` selects the encrypted field from a choice (e.g. the completion `text` or the
    chat `message.content`). Returns the raw parsed response.
    """
    client_private_key = x25519.X25519PrivateKey.generate()
    client_public_key = client_private_key.public_key()
    client_public_key_b64 = base64.b64encode(client_public_key.public_bytes_raw()).decode("utf-8")

    response = runtime_sm_client.invoke_endpoint(
        EndpointName=MODEL_NAME,
        ContentType="application/json",
        Body=json.dumps(body),
        CustomAttributes=client_public_key_b64,  # client public key (headers are stripped by InvokeEndpoint)
    )

    server_public_key = x25519.X25519PublicKey.from_public_bytes(base64.b64decode(response["CustomAttributes"]))
    shared_aes_key = derive_shared_aes_key(client_private_key, server_public_key)

    output = json.loads(response["Body"].read().decode("utf8"))
    for choice in output["choices"]:
        print(decrypt_str(ciphertext_of(choice), shared_aes_key=shared_aes_key))
    return output

## 4. Real-time inference

### A. Chat request

In [ ]:
PROMPT = "Why is the greenland shark the best shark?"
chat_request: dict[str, Any] = {
    "messages": [{"role": "user", "content": PROMPT}],
    "max_tokens": 150,
}

invoke_and_decrypt(chat_request, ciphertext_of=lambda choice: choice["message"]["content"])

### B. Completions

In [ ]:
completion_request: dict[str, Any] = {
    "prompt": PROMPT,
    "max_tokens": 120,
}

invoke_and_decrypt(completion_request, ciphertext_of=lambda choice: choice["text"])

### C. Generate from embeddings (input protection)

With the Stained Glass Transform, the prompt is sent as **embeddings**, never plaintext. In production those protected embeddings come from the Stained Glass Proxy. Here we send **random** embeddings of the correct shape purely to exercise the `prompt_embeds` path end to end — the generated text will be arbitrary, but it round-trips through input + output protection.

The embeddings' last dimension **must** equal the model's hidden size, or vLLM rejects the request.

In [ ]:
import io

import torch

HIDDEN_SIZE = 4096  # NVIDIA-Nemotron-3-Super-120B hidden_size (config.json); set to YOUR model's hidden size
NUM_TOKENS = 8

# Stand-in for SGT-protected embeddings from the Stained Glass Proxy.
embedding_inputs = torch.randn(NUM_TOKENS, HIDDEN_SIZE, dtype=torch.bfloat16)

buffer = io.BytesIO()
torch.save(embedding_inputs, buffer)
buffer.seek(0)
embedding_request_data = base64.b64encode(buffer.read()).decode("utf-8")

In [ ]:
# Send prompt_embeds alone — vLLM rejects an empty "prompt" alongside prompt_embeds.
request_w_embedding_data: dict[str, Any] = {
    "prompt_embeds": embedding_request_data,
    "max_tokens": 60,
}

invoke_and_decrypt(request_w_embedding_data, ciphertext_of=lambda choice: choice["text"])

## 5. Clean-up

Terminate the endpoint and delete the model to avoid further charges.

In [ ]:
model.sagemaker_session.delete_endpoint(MODEL_NAME)
model.sagemaker_session.delete_endpoint_config(MODEL_NAME)

In [ ]:
model.delete_model()